In [5]:
import torch
from PIL import Image
import os
import numpy as np
from transformers import CLIPProcessor, CLIPModel
import matplotlib.pyplot as plt
import yaml
from cleanfid import fid
import lpips
import torchvision.transforms as T
import itertools


with open(os.path.expanduser("~/prog/config.yaml"), "r") as f:
    config = yaml.safe_load(f)

In [ ]:
DEVICE = "cuda"
REF_IMAGES_DIR = config["paths"]["instance_images_dir"]  # Path to the reference image
LORA_GEN_DIR = config["paths"]["evaluation_dir"] + "/metrics/preservability/images_lora"  # Directory to save images generated with Lora
BASE_GEN_DIR = config["paths"]["evaluation_dir"] + "/metrics/preservability/images_base"  # Directory to save images generated without Lora

LORA_CAT = os.path.join(LORA_GEN_DIR, "cats")
BASE_CAT = os.path.join(BASE_GEN_DIR, "cats")

prompts = [
    "A photo of a cat",
    "A photo of a dog",
    "A photo of a rabbit",
    "A photo of a fox",
    "A photo of a lion",
    "A photo of a tiger",
    "A mountain landscape",
    "A portrait of a person",
    "A city skyline",
    "A bowl of fruit",
    "A car on a road"
]

In [9]:
model_id = "openai/clip-vit-large-patch14"  # Model version: ViT Large (24 layers) with patch14 (each patch is 14x14 pixels)
model = CLIPModel.from_pretrained(model_id).to(DEVICE)  # Load the CLIP model (pretrained) and move it to the specified device (GPU or CPU)
processor = CLIPProcessor.from_pretrained(model_id)  # Load the CLIP processor to automatically preprocess images and text for the model

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 781.53it/s] 
CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## CLIP-T Metric
Here we compare the Image with the prompt. If it's high, the model understood the prompt. 
We want the values of LoRA Model and Base Model are as similar as possible

In [ ]:
def compute_clip_t(image_path, text_prompt):
    img = Image.open(image_path).convert("RGB")
    
    # Processor accepts both text and images, and returns tensors ready for the model
    inputs = processor(
        text=[text_prompt], 
        images=[img], 
        return_tensors="pt", 
        padding=True
    ).to(DEVICE)
    
    with torch.no_grad():
        outputs = model(**inputs)
        
        # We extract the image and text embeddings from the model's output
        image_embeds = outputs.image_embeds
        text_embeds = outputs.text_embeds
        
        image_embeds /= image_embeds.norm(p=2, dim=-1, keepdim=True)
        text_embeds /= text_embeds.norm(p=2, dim=-1, keepdim=True)
        
        # Cosine similarity is computed as the dot product of the normalized embeddings
        similarity = (text_embeds @ image_embeds.T).item()
        
    return similarity


def evaluate_clipt(folder_path, nprompt):
    files = []
    p = prompts[nprompt]
    n = nprompt+1
    if n < 10: n = f"0{n}"
    for f in os.listdir(folder_path):
        if f.startswith(f"image_{n}"):
            files.append(os.path.join(folder_path, f))

    scores = []
    for file in files:
        score = compute_clip_t(file, p)
        scores.append(score)

    return np.mean(scores), np.std(scores)


results = []
print(f"{'Prompt':<25} | {'Base':<15} | {'LoRA':<15} | {'Drift (%)':<10}")
print("-" * 75)
for i, prompt in enumerate(prompts):
    mean_base, std_base = evaluate_clipt(BASE_GEN_DIR, i)
    mean_lora, std_lora = evaluate_clipt(LORA_GEN_DIR, i)

    current_drift = (1 - (mean_lora / mean_base)) * 100 if mean_base != 0 else 0

    results.append({
        "prompt": prompt,
        "base": mean_base,
        "lora": mean_lora,
        "drift": current_drift
    })

    print(f"{prompt[:25]:<25} | {mean_base:.4f} ± {std_base:.4f} | {mean_lora:.4f} ± {std_lora:.4f} | {current_drift:.2f}%")

avg_base = np.mean([r["base"] for r in results])
avg_lora = np.mean([r["lora"] for r in results])
avg_drift = (1 - (avg_lora / avg_base)) * 100

print("-"*75)
print(f"Average CLIP-T Base: {avg_base:.4f}, Average CLIP-T LoRA: {avg_lora:.4f}, Average Drift: {avg_drift:.2f}%")

Prompt                    | Base            | LoRA            | Drift (%) 
---------------------------------------------------------------------------
A photo of a cat          | 0.2561 ± 0.0055 | 0.2576 ± 0.0070 | -0.60%
A photo of a dog          | 0.2362 ± 0.0100 | 0.2469 ± 0.0079 | -4.52%
A photo of a rabbit       | 0.2536 ± 0.0082 | 0.2682 ± 0.0028 | -5.76%
A photo of a fox          | 0.2688 ± 0.0058 | 0.2706 ± 0.0058 | -0.67%
A photo of a lion         | 0.2538 ± 0.0056 | 0.2562 ± 0.0100 | -0.94%
A photo of a tiger        | 0.2576 ± 0.0066 | 0.2649 ± 0.0083 | -2.84%
A mountain landscape      | 0.2443 ± 0.0082 | 0.2425 ± 0.0093 | 0.75%
A portrait of a person    | 0.2175 ± 0.0074 | 0.2299 ± 0.0060 | -5.72%
A city skyline            | 0.2416 ± 0.0122 | 0.2391 ± 0.0133 | 1.05%
A bowl of fruit           | 0.2516 ± 0.0112 | 0.2568 ± 0.0076 | -2.07%
A car on a road           | 0.2286 ± 0.0089 | 0.2334 ± 0.0073 | -2.12%
----------------------------------------------------------------------

## FID Metric
Here we compare a group of images (e.g. 50 images generated by LoRA Model) with another group of images (e.g. images generated by Base Model)

In [ ]:
folder_base = BASE_CAT
folder_lora = LORA_CAT

# Compute FID
score_fid = fid.compute_fid(folder_base, folder_lora, device=torch.device("cuda"))

print(f"FID Score: {score_fid:.4f}")
print("FID computation completed. Note: the more value is close to 0, the more similar the two sets of images are.")

/homes/saresta/cvcs2026/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 6, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


compute FID between two folders
Found 50 images in the folder /homes/saresta/prog/evaluation/metrics/preservability/images_base/cats


FID cats : 100%|██████████| 2/2 [00:05<00:00,  2.66s/it]


Found 50 images in the folder /homes/saresta/prog/evaluation/metrics/preservability/images_lora/cats


FID cats : 100%|██████████| 2/2 [00:04<00:00,  2.19s/it]


FID Score: 100.1438
Nota: Più il valore è vicino a 0, più il tuo LoRA ha preservato il concetto originale.


## LPIPS Metric
Here we compute the Diversity. We give a group of images (e.g. 50 images generated by LoRA Model), and LPIPS measures how different the images are.

In [ ]:
loss_fn_alex = lpips.LPIPS(net='alex').to("cuda")

def load_and_preprocess(path):
    img = Image.open(path).convert("RGB")
    transform = T.Compose([
        T.Resize((256, 256)),
        T.ToTensor(),
        T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) 
    ])
    return transform(img).unsqueeze(0).to("cuda")

def compute_internal_diversity(folder_path):
    files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.png')]
    if len(files) < 2: return 0
    
    distances = []
    
    for path_a, path_b in itertools.combinations(files, 2):
        img_a = load_and_preprocess(path_a)
        img_b = load_and_preprocess(path_b)
        
        with torch.no_grad():
            dist = loss_fn_alex(img_a, img_b)
            distances.append(dist.item())
            
    return sum(distances) / len(distances)

# Esecuzione
div_base = compute_internal_diversity(BASE_CAT)
div_lora = compute_internal_diversity(LORA_CAT)

print(f"Internal Diversity (LPIPS) Base: {div_base:.4f}")
print(f"Internal Diversity (LPIPS) LoRA: {div_lora:.4f}")

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/homes/saresta/cvcs2026/venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/homes/saresta/cvcs2026/venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /homes/saresta/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth
100%|██████████| 233M/233M [00:00<00:00, 262MB/s]  


Loading model from: /homes/saresta/cvcs2026/venv/lib/python3.11/site-packages/lpips/weights/v0.1/alex.pth
Internal Diversity (LPIPS) Base: 0.5934
Internal Diversity (LPIPS) LoRA: 0.5515
